**VitalSign Real-Time AI Sunum Alanı**

In [ ]:
import cv2
import numpy as np
import os
import mediapipe as mp
import pyttsx3
import threading
import time
import tkinter as tk
from tkinter import ttk
from PIL import Image, ImageTk
from tensorflow.keras.models import load_model
import queue # Veri kuyrukları için yeni kütüphane

# PREVENT HAYALET RESİM TKINTER CRASH
try:
    root.destroy()
except:
    pass

# 1. LOAD MODEL
MODEL_PATH = 'vitalsign_local_model.keras'
model = load_model(MODEL_PATH)

actions = np.array([
    'help', 'doctor', 'emergency', 
    'pain', 'medicine', 'hospital', 'blood', 'breathe',
    'head', 'heart', 'arm', 'leg', 'allergy', 'bad',
    'fever', 'cough', 'dizzy', 'nausea',
    'today', 'yesterday', 'much', 'little',
    'diabetic', 'surgery', 'accident',
    'yes', 'no', 'hello', 'thanks', 'how-are-you'
])

# ==================== THREAD-SAFE GÜVENLİK KUYRUKLARI ====================
speech_queue = queue.Queue()  # Ses motoru için kuyruk
gui_queue = queue.Queue()     # Arayüz güncellemeleri için kuyruk

# 2. STABLE TTS WORKER (Tek bir sabit kanaldan seslendirir)
def tts_worker():
    try:
        engine = pyttsx3.init()
        engine.setProperty('rate', 145)
    except Exception as e:
        print(f"TTS Initialization Error: {e}")
        return

    while True:
        text = speech_queue.get()
        if text is None:
            break
        try:
            engine.say(text)
            engine.runAndWait()
        except Exception as e:
            print(f"TTS Playback Error: {e}")
        finally:
            speech_queue.task_done()

threading.Thread(target=tts_worker, daemon=True).start()

def speak(text):
    speech_queue.put(text)

# 3. MEDICAL NLP DICTIONARY
sentence_dict = {
    ('head', 'pain'): "I have a severe headache.",
    ('arm', 'pain'): "My arm hurts.",
    ('fever', 'today'): "I have a high fever today.",
    ('dizzy', 'much'): "I feel very dizzy right now.",
    ('diabetic', 'yes'): "Yes, I am a diabetic patient.",
    ('allergy', 'yes'): "Yes, I have an allergy.",
    ('help', 'surgery'): "I need assistance after my surgery.",
    ('accident', 'help'): "There was an accident, please help!",
    ('hello', 'thanks'): "Hello, thank you very much.",
    ('blood', 'help'): "CRITICAL: Blood is needed urgently!", 
    ('blood', 'bad'): "The blood pressure or bleeding looks bad.",
    ('heart', 'bad'): "My heart feels bad."
}

# 4. MEDIAPIPE CORE
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils
holistic = mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5)

def extract_landmarks(results):
    lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3)
    rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)
    lh_xy = lh.reshape(-1, 3)[:, :2].flatten()
    rh_xy = rh.reshape(-1, 3)[:, :2].flatten()
    face = np.array([[res.x, res.y] for res in results.face_landmarks.landmark[:12]]).flatten() if results.face_landmarks else np.zeros(12*2)
    return np.concatenate([lh_xy, rh_xy, face])

# GLOBAL VARIABLES & EMERGENCE LOGS
sequence = []        
detected_words = []   
final_sentence = "Waiting for patient's signs..."  
doctor_message = "Examination started. The device is listening to the patient..."
medical_history_log = [] 
current_frame = None
is_running = True

# 5. TKINTER ULTIMATE MEDIKAL GUI INTERFACE
root = tk.Tk()
root.title("VitalSign - Dual-Way Clinical Communication Ecosystem")
root.geometry("1200x750")
root.configure(bg="#f1f5f9")

# Header Frame
header_frame = tk.Frame(root, bg="#0f172a", height=65)
header_frame.pack(fill="x")
header_label = tk.Label(header_frame, text=" 🩺 VitalSign - Dual-Way Hospital Communication Platform", font=("Helvetica", 16, "bold"), fg="white", bg="#0f172a")
header_label.pack(pady=15)

main_frame = tk.Frame(root, bg="#f1f5f9")
main_frame.pack(fill="both", expand=True, padx=15, pady=15)

# ----------------- LEFT SIDE (CAMERA & PATIENT SCREEN) -----------------
left_container = tk.Frame(main_frame, bg="#f1f5f9")
left_container.pack(side="left", fill="both", expand=True, padx=5)

video_frame = tk.LabelFrame(left_container, text=" Live Patient Camera Stream ", font=("Helvetica", 11, "bold"), bg="white", bd=1)
video_frame.pack(fill="both", expand=True)

video_label = tk.Label(video_frame, bg="black")
video_label.pack(fill="both", expand=True, padx=5, pady=5)

doctor_live_billboard = tk.Label(left_container, text=f"💬 DOCTOR'S MESSAGE: {doctor_message}", font=("Helvetica", 13, "bold"), fg="white", bg="#2563eb", height=2, wraplength=700)
doctor_live_billboard.pack(fill="x", pady=(10, 0))

# ----------------- RIGHT SIDE (DIAGNOSTICS & DOCTOR CONTROLS) -----------------
sidebar_frame = tk.Frame(main_frame, bg="#f1f5f9", width=400)
sidebar_frame.pack(side="right", fill="both", padx=5)

# 1. AI Diagnostics
debug_frame = tk.LabelFrame(sidebar_frame, text=" 📊 AI Diagnostics (Real-Time) ", font=("Helvetica", 10, "bold"), bg="white", bd=1)
debug_frame.pack(fill="x", pady=(0, 8))

live_prediction_label = tk.Label(debug_frame, text="Current Prediction: None (0%)", font=("Helvetica", 10), fg="#64748b", bg="white", anchor="w")
live_prediction_label.pack(fill="x", padx=10, pady=4)

queue_label = tk.Label(debug_frame, text="Word Queue: Empty", font=("Helvetica", 10, "bold"), fg="#3b82f6", bg="white", anchor="w", wraplength=350)
queue_label.pack(fill="x", padx=10, pady=4)

# 2. Sensitivity Threshold
threshold_frame = tk.LabelFrame(sidebar_frame, text=" 🎛️ Sensitivity Threshold Control ", font=("Helvetica", 10, "bold"), bg="white", bd=1)
threshold_frame.pack(fill="x", pady=4)
thresh_val_label = tk.Label(threshold_frame, text="Confidence Threshold: %80", font=("Helvetica", 9), bg="white")
thresh_val_label.pack(pady=(4,0))

def on_thresh_change(val):
    thresh_val_label.config(text=f"Confidence Threshold: %{int(float(val)*100)}")

thresh_slider = ttk.Scale(threshold_frame, from_=0.40, to=0.95, value=0.80, orient="horizontal", command=on_thresh_change)
thresh_slider.pack(fill="x", padx=15, pady=8)

# 3. Patient Translated Output
result_frame = tk.LabelFrame(sidebar_frame, text=" 🔊 Patient's Translated Statement ", font=("Helvetica", 11, "bold"), bg="white", bd=1, height=130)
result_frame.pack(fill="x", pady=4)
result_frame.pack_propagate(False) 

sentence_display = tk.Label(result_frame, text=final_sentence, font=("Helvetica", 12, "bold"), fg="#475569", bg="white", wraplength=350, justify="center")
sentence_display.pack(fill="both", expand=True, padx=10)

# 4. Doctor Communication Input
doctor_panel_frame = tk.LabelFrame(sidebar_frame, text=" 💬 Doctor's Communication Input Panel ", font=("Helvetica", 11, "bold"), bg="white", bd=1)
doctor_panel_frame.pack(fill="x", pady=4)

doctor_entry = tk.Entry(doctor_panel_frame, font=("Helvetica", 11), bd=1, relief="solid")
doctor_entry.pack(fill="x", padx=10, pady=8)

def send_doctor_msg(event=None):
    global doctor_message
    msg = doctor_entry.get().strip()
    if msg:
        doctor_message = msg
        doctor_live_billboard.config(text=f"💬 DOCTOR'S MESSAGE: {doctor_message}", bg="#0d9488")
        medical_history_log.append(("DOCTOR NOTE", msg, time.strftime('%H:%M:%S')))
        doctor_entry.delete(0, tk.END)

send_msg_btn = tk.Button(doctor_panel_frame, text="✉️ Send Message to Patient Screen", font=("Helvetica", 10, "bold"), fg="white", bg="#0d9488", bd=0, cursor="hand2", command=send_doctor_msg, height=1)
send_msg_btn.pack(fill="x", padx=10, pady=(0, 8))
doctor_entry.bind("<Return>", send_doctor_msg)

# 5. HIGH-END HTML REPORT EXPORTER
def export_medical_report():
    try:
        report_filename = "VitalSign_Medical_Report.html"
        
        html_content = f"""<!DOCTYPE html>
        <html>
        <head>
            <meta charset="utf-8">
            <title>VitalSign AI - Examination Report</title>
            <style>
                body {{ font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; background-color: #f8fafc; color: #1e293b; margin: 40px; }}
                .report-card {{ max-width: 800px; margin: 0 auto; background: white; padding: 30px; border-radius: 12px; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1); border-top: 8px solid #0f172a; }}
                .header {{ text-align: center; border-bottom: 2px solid #e2e8f0; padding-bottom: 20px; margin-bottom: 25px; }}
                .header h1 {{ margin: 0; color: #0f172a; font-size: 24px; }}
                .header p {{ margin: 5px 0 0 0; color: #64748b; font-size: 14px; }}
                .meta-grid {{ display: grid; grid-template-columns: 1fr 1fr; gap: 15px; margin-bottom: 30px; background: #f1f5f9; padding: 15px; border-radius: 8px; font-size: 14px; }}
                .meta-grid-item {{ font-weight: 500; }}
                .meta-grid-item span {{ color: #64748b; font-weight: 400; }}
                table {{ width: 100%; border-collapse: collapse; margin-top: 15px; }}
                th {{ background-color: #0f172a; color: white; text-align: left; padding: 12px; font-size: 14px; }}
                td {{ padding: 12px; border-bottom: 1px solid #e2e8f0; font-size: 14px; }}
                tr:nth-child(even) {{ background-color: #f8fafc; }}
                .badge-patient {{ background: #dbeafe; color: #1e40af; padding: 4px 8px; border-radius: 4px; font-weight: 600; font-size: 12px; }}
                .badge-doctor {{ background: #ccfbf1; color: #115e59; padding: 4px 8px; border-radius: 4px; font-weight: 600; font-size: 12px; }}
                .signature {{ margin-top: 50px; border-top: 1px dashed #cbd5e1; padding-top: 20px; text-align: right; color: #64748b; font-size: 14px; }}
            </style>
        </head>
        <body>
            <div class="report-card">
                <div class="header">
                    <h1>🩺 VITALSIGN MEDICAL AI PLATFORM</h1>
                    <p>Real-Time Patient-Doctor Examination Summary Log</p>
                </div>
                <div class="meta-grid">
                    <div class="meta-grid-item">Examination Date: <span>{time.strftime('%Y-%m-%d')}</span></div>
                    <div class="meta-grid-item">System Status: <span>Active / Operational</span></div>
                    <div class="meta-grid-item">Export Time: <span>{time.strftime('%H:%M:%S')}</span></div>
                    <div class="meta-grid-item">Gateway Mode: <span>Dual-Way Tkinter Thread</span></div>
                </div>
                <h3>📄 Clinical Examination Dialogue History</h3>
                <table>
                    <tr>
                        <th style="width: 20%;">Time</th>
                        <th style="width: 25%;">Sender</th>
                        <th style="width: 55%;">Statement / Statement Details</th>
                    </tr>
        """
        
        if not medical_history_log:
            html_content += """<tr><td colspan="3" style="text-align:center; color:#64748b;">No dialogue logs created in this session yet.</td></tr>"""
        else:
            for sender, msg, log_time in medical_history_log:
                badge = "badge-patient" if "PATIENT" in sender else "badge-doctor"
                html_content += f"""
                <tr>
                    <td>{log_time}</td>
                    <td><span class="{badge}">{sender}</span></td>
                    <td>{msg}</td>
                </tr>
                """
                
        html_content += f"""
                </table>
                <div class="signature">
                    <p>M.D. Final Signature: ............................................</p>
                </div>
            </div>
        </body>
        </html>
        """
        
        with open(report_filename, "w", encoding="utf-8") as f:
            f.write(html_content)
        
        sentence_display.config(text=f"💾 SUCCESS:\n'VitalSign_Medical_Report.html' generated successfully! Double click to open on Edge/Chrome.", fg="#10b981")
    except Exception as e:
        sentence_display.config(text=f"Export Error: {e}", fg="#ef4444")

action_buttons_frame = tk.Frame(sidebar_frame, bg="#f1f5f9")
action_buttons_frame.pack(fill="x", pady=5)

export_btn = tk.Button(action_buttons_frame, text="💾 Export Medical Report", font=("Helvetica", 11, "bold"), fg="white", bg="#1e293b", bd=0, cursor="hand2", command=export_medical_report, height=2)
export_btn.pack(fill="x", pady=2)

def reset_memory():
    global detected_words, final_sentence
    detected_words = []
    final_sentence = "System reset. Waiting for signs..."
    sentence_display.config(text=final_sentence, fg="#475569")
    queue_label.config(text="Word Queue: Empty")

reset_btn = tk.Button(action_buttons_frame, text="🔄 Reset Memory", font=("Helvetica", 10), fg="white", bg="#ef4444", bd=0, cursor="hand2", command=reset_memory, height=1)
reset_btn.pack(fill="x", pady=2)

# ==================== KİLİTLENMEYİ ÖNLEYEN GÜVENLİ GUI YÖNETİCİSİ ====================
def check_gui_queue():
    """Arka plan thread'lerinden gelen ekran güncelleme isteklerini ana döngüde güvenle işler."""
    while not gui_queue.empty():
        task, data = gui_queue.get()
        try:
            if task == "PREDICTION":
                live_prediction_label.config(text=data)
            elif task == "QUEUE":
                queue_label.config(text=data)
            elif task == "SENTENCE":
                sentence_display.config(text=data["text"], fg=data["color"])
        except Exception as e:
            print(f"GUI Sync Error: {e}")
        finally:
            gui_queue.task_done()
    root.after(50, check_gui_queue) # Her 50 milisaniyede bir kuyruğu kontrol et

# 6. BACKROUND THREAD WORKER FOR AI
def ai_prediction_worker():
    global sequence, detected_words, final_sentence, current_frame, is_running
    
    while is_running:
        if current_frame is not None:
            frame_copy = current_frame.copy()
            rgb_frame = cv2.cvtColor(frame_copy, cv2.COLOR_BGR2RGB)
            results = holistic.process(rgb_frame)
            
            keypoints = extract_landmarks(results)
            sequence.append(keypoints)
            sequence = sequence[-30:]
            
            current_threshold = thresh_slider.get()
            
            if len(sequence) == 30:
                res = model.predict(np.expand_dims(sequence, axis=0), verbose=0)[0]
                predicted_index = np.argmax(res)
                prob = res[predicted_index]
                word = actions[predicted_index]
                
                # Arayüze doğrudan dokunma! İsteği güvenli kuyruğa at
                gui_queue.put(("PREDICTION", f"Current Prediction: {word.upper()} (%{prob*100:.1f})"))
                
                if prob > current_threshold:
                    if len(detected_words) == 0 or detected_words[-1] != word:
                        detected_words.append(word)
                        detected_words = detected_words[-3:]
                        
                        gui_queue.put(("QUEUE", f"Word Queue: {' -> '.join([w.upper() for w in detected_words])}"))
                        
                        if len(detected_words) >= 2:
                            for i in range(len(detected_words) - 1):
                                potential_combo = (detected_words[i], detected_words[i+1])
                                if potential_combo in sentence_dict:
                                    final_sentence = sentence_dict[potential_combo]
                                    
                                    medical_history_log.append(("PATIENT COMPLAINT", final_sentence, time.strftime('%H:%M:%S')))
                                    
                                    color = "#10b981" if "CRITICAL" not in final_sentence else "#ef4444"
                                    gui_queue.put(("SENTENCE", {"text": final_sentence, "color": color}))
                                    
                                    speak(final_sentence)
                                    detected_words = []
                                    break
                            if len(detected_words) == 3:
                                detected_words.pop(0)
        time.sleep(0.03)

# 7. FOREGROUND THREAD FOR LIVE VIDEO
cap = cv2.VideoCapture(0)

def update_frame():
    global current_frame
    
    ret, frame = cap.read()
    if ret:
        current_frame = frame.copy()
        
        cv2_image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        img = Image.fromarray(cv2_image)
        imgtk = ImageTk.PhotoImage(image=img)
        video_label.imgtk = imgtk
        video_label.configure(image=imgtk)
        
    video_label.after(15, update_frame)

# Launch Ecosystem
threading.Thread(target=ai_prediction_worker, daemon=True).start()
update_frame()
root.after(50, check_gui_queue) # Güvenlik köprüsünü aktif et

def on_closing():
    global is_running
    is_running = False
    cap.release()
    speech_queue.put(None) # Ses döngüsünü kapat
    root.destroy()

root.protocol("WM_DELETE_WINDOW", on_closing)
root.mainloop()